# 構造化出力 GP: HigherOrderGP と LatentKroneckerGP

この Notebook では、合成スペクトルデータを使って構造化出力向けの2つの robotorchan モデルを比較します。

- `HigherOrderGP`: 出力を tensor として扱い、分離可能な出力構造を利用します。
- `LatentKroneckerGP`: 設計変数 `X` と、波長・時間・位置などの明示的な出力座標 `T` の積空間をモデル化します。

## このNotebookで学ぶこと

- スカラー出力GPと**構造化出力GP**の違い
- `train_X`、`train_T`、`train_Y` のshapeと対応関係
- HigherOrderGPとLatentKroneckerGPのモデル化の違い
- スペクトル全体の予測、出力座標の補間、候補提案
- Mixed版のカテゴリ設計変数の扱い

構造化出力GPは、1つの設計条件から1つのスカラーではなく、**スペクトル・画像・時系列などの配列**が得られる場合に利用します。

## 1. これらのモデルを使う場面

`HigherOrderGP` は、各実験から画像・スペクトルグリッド・その他の tensor-valued response が得られ、その tensor shape 自体に意味がある場合に向きます。

`LatentKroneckerGP` は、出力が波長・時間・位置などの座標 `T` で自然に添字付けされ、その軸に沿って明示的に補間・予測したい場合に向きます。

In [ ]:
import matplotlib.pyplot as plt
import torch
from robotorchan.models import HigherOrderGP, LatentKroneckerGP

torch.set_default_dtype(torch.double)
torch.manual_seed(0)

### 出力構造をモデル化する理由

例えば製造条件 $x=(x_1,x_2)$ を1回評価すると、32波長のスペクトル $y(x,t_1),\dots,y(x,t_{32})$ が得られます。波長ごとに独立した32個のGPを作ると、波長間の滑らかな関係を十分に利用できない場合があります。

**HigherOrderGP**は出力テンソルの構造を利用します。**LatentKroneckerGP**は入力条件 `X` と波長座標 `T` の双方に関する共分散構造を使います。どちらが適するかは出力の物理的な構造に依存します。

## 2. 合成スペクトルデータ

`X` は2つのプロセス条件を表します。各条件から、波長軸 `T` 上で観測される1本のスペクトルが得られるとします。

In [ ]:
n_train = 12
n_wavelengths = 32
train_X = torch.rand(n_train, 2)
train_T = torch.linspace(0.0, 1.0, n_wavelengths).unsqueeze(-1)
def spectrum(X, T):
    x1 = X[..., 0].unsqueeze(-1)
    x2 = X[..., 1].unsqueeze(-1)
    t = T.squeeze(-1)
    peak1 = torch.exp(-0.5 * ((t - (0.25 + 0.15 * x1)) / 0.07) ** 2)
    peak2 = 0.7 * torch.exp(-0.5 * ((t - (0.70 - 0.12 * x2)) / 0.10) ** 2)
    baseline = 0.15 * x1 + 0.08 * x2
    return peak1 + peak2 + baseline
train_Y = spectrum(train_X, train_T)
train_Y = train_Y + 0.02 * torch.randn_like(train_Y)
print(train_X.shape)
print(train_T.shape)
print(train_Y.shape)

In [ ]:
plt.figure(figsize=(8, 4))
for i in range(min(6, n_train)):
    plt.plot(train_T.squeeze(-1), train_Y[i], alpha=0.8)
plt.xlabel("normalized wavelength")
plt.ylabel("response")
plt.title("Synthetic training spectra")
plt.show()

### データのshapeと対応関係

- `train_X.shape == (12, 2)`：12実験、各実験に2つの設計変数
- `train_T.shape == (32, 1)`：各実験で共通の32波長座標
- `train_Y.shape == (12, 32)`：各実験の32波長応答

ここでは**すべての実験で同じ波長グリッドを観測**しています。出力座標を別のグリッドで予測できることと、学習時に任意の欠損波長を直接扱えることは別です。

## 3. HigherOrderGP

HOGP では response tensor をそのまま `train_Y` として渡します。1次元スペクトルも構造化出力ですが、HOGP は画像や時空間グリッドなど、より高階の配列で特に自然です。

学習時には `model.fit(step_limit=75)` を利用します。必要な fast-solve 設定はモデルが内部で有効にします（内部でBoTorchのtorch optimizerを使用）。

In [ ]:
hogp = HigherOrderGP(train_X=train_X, train_Y=train_Y)
print("supports_mll:", hogp.supports_mll)
print("raw_train_X:", hogp.raw_train_X.shape)
print("raw_train_Y:", hogp.raw_train_Y.shape)
print("raw_train_Yvar:", hogp.raw_train_Yvar)
hogp.fit(step_limit=75)
hogp.eval()

### HigherOrderGPの予測

`posterior(test_X)` は、指定した設計条件に対する**スペクトル全体**の事後分布を返します。`posterior.mean` は波長ごとの予測平均です。出力軸を無条件にflattenすると、どの軸が波長なのか分からなくなるため、shapeを確認してから集約してください。

学習に必要な fast-solve 設定は `HigherOrderGP.fit()` が内部で管理します。Notebook側で `_fast_solves` を import する必要はありません。この設定は内部 API に依存するため、今後の上流API更新時には実装側で互換性を確認します。

In [ ]:
test_X = torch.tensor([[0.65, 0.30]])
hogp_posterior = hogp.posterior(test_X)
hogp_mean = hogp_posterior.mean.squeeze(0).detach()
print("HOGP posterior mean shape:", hogp_mean.shape)

## 4. LatentKroneckerGP

`LatentKroneckerGP` では出力座標を `train_T` として明示的に渡します。そのため robotorchan は共通 raw training tensor に加えて `raw_train_T` も保持します。

`LatentKroneckerGP.fit()` は学習時に必要な `use_iterative_methods()` を内部で有効化するため、Notebook側で `with` を書く必要はありません。`posterior()` と事後平均・分散・サンプリングでも反復計算設定は内部で管理されます。
**重要：出力スケール** `LatentKroneckerGP` の既定の `outcome_transform` は観測値を標準化します。robotorchan の `posterior()` は予測平均・分散を観測スケールに戻します。BoTorch のパスワイズ事後サンプルはすでに逆変換されるため、ラッパーではサンプリング経路を再変換しません。Notebook 側で `outcome_transform` を手動で適用する必要はありません。


In [ ]:
lkgp = LatentKroneckerGP(train_X=train_X, train_T=train_T, train_Y=train_Y)
print("supports_mll:", lkgp.supports_mll)
print("raw_train_X:", lkgp.raw_train_X.shape)
print("raw_train_T:", lkgp.raw_train_T.shape)
print("raw_train_Y:", lkgp.raw_train_Y.shape)
lkgp.fit()
lkgp.eval()

### LatentKroneckerGPの予測

`posterior(test_X, train_T)` は、設計条件 `X` と波長座標 `T` の組合せに対する事後分布です。波長ごとの予測分散を取り出せるので、どの波長領域でモデルの不確実性が高いかも確認できます。

以下の95%区間はモデル上の**潜在スペクトルの近似事後区間**です。観測ノイズ込みの将来測定値の予測区間や、頻度論的な被覆保証とは異なります。
ここで定義する `latent_original_moments()` は、モデルが元の観測単位で返した平均・分散を `q × t` に整形するだけの表示用ヘルパーです。


In [ ]:
def latent_original_moments(model, X, T):
    """Return (mean, variance) in the observed Y units, shaped (q, t)."""
    with torch.no_grad():
        posterior = model.posterior(X, T)
        shape = (*X.shape[:-1], T.shape[-2])
        mean = posterior.mean.reshape(shape)
        variance = posterior.variance.reshape(shape)
    assert mean.shape == (*X.shape[:-1], T.shape[-2])
    assert torch.isfinite(mean).all()
    assert torch.isfinite(variance).all()
    return mean, variance.clamp_min(0)


lkgp_mean_2d, lkgp_var_2d = latent_original_moments(lkgp, test_X, train_T)
lkgp_mean = lkgp_mean_2d.squeeze(0)
lkgp_std = lkgp_var_2d.sqrt().squeeze(0)
print("LatentKronecker posterior mean shape:", lkgp_mean.shape)
print("Original Y range:", float(train_Y.min()), float(train_Y.max()))
print("LatentKronecker mean range:", float(lkgp_mean.min()), float(lkgp_mean.max()))
print("LatentKronecker RMSE at test X:",
      float(torch.sqrt(torch.mean((lkgp_mean - spectrum(test_X, train_T).squeeze(0)) ** 2))))


### 元スケールでの再確認

学習時の `train_Y` と予測値を同じ単位で比較するため、まず学習点の再現誤差と、別の設計条件の真値との誤差を表示します。負値が残っても GP のガウス分布は非負制約を保証しないため、それだけで不具合とは判定しません。

なお、学習点での適合度と未観測点への汎化性能は異なります。比較値が大きい場合は、学習の収束、ノイズ、カーネル・モデル仮定も確認してください。

In [ ]:
training_mean, _ = latent_original_moments(lkgp, train_X, train_T)
train_rmse = ((training_mean - train_Y).square().mean()).sqrt()
validation_X = torch.tensor([[0.20, 0.75], [0.80, 0.15]], dtype=train_X.dtype)
validation_mean, _ = latent_original_moments(lkgp, validation_X, train_T)
validation_truth = spectrum(validation_X, train_T)
validation_rmse = ((validation_mean - validation_truth).square().mean()).sqrt()
print("Training RMSE (original Y units):", float(train_rmse))
print("Unseen-X RMSE (original Y units):", float(validation_rmse))
assert training_mean.shape == train_Y.shape
assert validation_mean.shape == validation_truth.shape


## 5. 予測結果の比較

合成データなので真のスペクトルも表示できます。実データでは、どの構造化出力仮定が問題に合っているか、また明示的な `T` 軸が有用かどうかがモデル選択の重要な観点です。

In [ ]:
true_y = spectrum(test_X, train_T).squeeze(0)
plt.figure(figsize=(9, 4))
plt.plot(train_T.squeeze(-1), true_y, label="true")
plt.plot(train_T.squeeze(-1), hogp_mean, label="HigherOrderGP")
plt.plot(train_T.squeeze(-1), lkgp_mean, label="LatentKroneckerGP")
plt.fill_between(train_T.squeeze(-1), lkgp_mean - 1.96 * lkgp_std, lkgp_mean + 1.96 * lkgp_std, alpha=0.2, label="LatentKronecker 95% interval")
plt.xlabel("normalized wavelength")
plt.ylabel("response")
plt.legend()
plt.title("Structured-output posterior at one process condition")
plt.show()

### スペクトル比較の読み方

真のスペクトル、HigherOrderGP、LatentKroneckerGPの予測平均を同じ波長軸で比較します。ここでの比較は**1つの設計条件**における可視化であり、モデル全体の優劣を決めるベンチマークではありません。

実務では複数の未学習設計条件に対して、波長ごとの誤差や積分値など、目的に対応した評価指標を確認します。

## 6. LatentKroneckerGP で新しい T グリッドを評価

出力座標を明示的に持つ利点の1つは、学習時とは異なる波長グリッドでも posterior を評価できることです。

In [ ]:
test_T = torch.linspace(0.05, 0.95, 50).unsqueeze(-1)
new_T_mean, new_T_variance = latent_original_moments(lkgp, test_X, test_T)
print("new T:", test_T.shape)
print("original-scale posterior mean:", new_T_mean.shape)


### 追加実装例：学習時と異なる波長グリッドでの補間

`LatentKroneckerGP` は新しい `test_T` を指定できるので、学習に使っていない波長位置でも予測できます。以下では新しい50点グリッド上の予測平均と真の合成スペクトルを重ねます。

新しいグリッドの範囲は学習範囲内に収めています。**補間と外挿は難易度が異なる**ことに注意してください。

In [ ]:
interpolated_mean = new_T_mean.reshape(-1)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(test_T.squeeze(-1), spectrum(test_X, test_T).reshape(-1), "--", label="true spectrum")
ax.plot(test_T.squeeze(-1), interpolated_mean, label="predicted on new T grid")
ax.scatter(train_T.squeeze(-1), train_Y[0], s=10, alpha=0.35, label="example training spectrum")
ax.set_xlabel("normalized wavelength")
ax.set_ylabel("response")
ax.legend()
plt.show()

## 7. モデル選択

| 状況 | 推奨モデル |
|---|---|
| 画像や多次元グリッドなどの tensor-valued output | `HigherOrderGP` |
| 意味のある座標を持つスペクトル・時系列・空間プロファイル | `LatentKroneckerGP` |
| 新しい出力座標位置で評価したい | `LatentKroneckerGP` |
| 出力 tensor が複数の構造化軸を持つ | `HigherOrderGP` |

物理的に意味のある波長軸を持つスペクトルでは、波長を `T` として明示できる `LatentKroneckerGP` が解釈しやすいことがあります。画像や本当に高階の出力では `HigherOrderGP` がより自然です。

## 8. Bayesian Optimization で使う場合

どちらも構造化出力の surrogate として利用できますが、BO では通常、構造化 response からスカラーまたは低次元の objective を定義する必要があります。例えば、ピーク強度、スペクトル積分面積、target spectrum との距離、波長方向の最大応答、物理知識に基づくスコアなどです。

獲得関数を選ぶ前に、構造化出力から何を最適化するのかを明示してください。

### スペクトルをどうスカラー目的へ変換するか

スペクトル全体を予測しても、**何を最大化したいか**を定義しない限りBOの候補は選べません。このNotebookでは $u(x)=\frac{1}{32}\sum_{j=1}^{32}y(x,t_j)$ という平均強度を例にします。

他にも、特定波長のピーク強度、指定波長帯の積分、目標スペクトルとの誤差、複数の特徴量を用いた多目的最適化などが考えられます。平均強度はあくまで説明用です。

## スペクトル目的関数に基づく候補提案

スペクトルの全波長での平均応答を最大化する設計条件を、既存の学習済みモデルから提案します。出力全体をスカラーに集約した事後平均を候補スコアとして使います。これは離散候補集合上の探索であり、獲得関数の連続最適化とは区別します。


In [ ]:
candidate_X = torch.rand(64, train_X.shape[-1])
with torch.no_grad():
    hogp_scores = hogp.posterior(candidate_X).mean.reshape(64, -1).mean(dim=-1)
    lkgp_candidate_mean, _ = latent_original_moments(lkgp, candidate_X, train_T)
    lkgp_scores = lkgp_candidate_mean.mean(dim=-1)

for name, scores in [("HigherOrderGP", hogp_scores), ("LatentKroneckerGP", lkgp_scores)]:
    best_index = scores.argmax()
    candidate = candidate_X[best_index:best_index + 1]
    evaluated_spectrum = spectrum(candidate, train_T)
    evaluated_utility = evaluated_spectrum.mean()
    assert candidate.shape == (1, 2)
    assert torch.isfinite(evaluated_utility)
    print(name, "candidate:", candidate)
    print("predicted mean utility:", scores[best_index].item())
    print("evaluated mean utility:", evaluated_utility.item())


### 候補生成の位置付け

このNotebookは64点のランダムな候補集合を作り、各候補に対する**事後平均スペクトルの平均値**で順位付けしています。これは**予測平均の最大化（greedy探索）**であり、EIやUCBのような不確実性を積極的に利用する獲得関数ではありません。

`HigherOrderGP` と `LatentKroneckerGP` はそれぞれ独立に最良候補を選ぶため、提案点が異なる場合があります。実際のBOでは探索と活用のバランスや評価コストも検討してください。

## 9. Mixed構造化出力GPでの候補提案

設計変数の第3列をカテゴリ値0/1とし、スペクトルの平均応答を最大化します。HigherOrderGPの出力テンソルとLatentKroneckerGPの明示的なT軸を維持します。候補集合上のPosterior平均ランキングであり、獲得関数最適化ではありません。


### Mixed構造化出力の意味

Mixed版では連続設計変数2列にカテゴリ列を1つ追加します。カテゴリはtask IDではなく**選択可能な設計条件**です。`cat_dims=[2]` でカテゴリ列を指定し、0と1をそれぞれ含む候補集合を評価します。

`MixedLatentKroneckerGP.fit()` も通常版の実装を継承し、学習・予測時の iterative methods を内部で管理します。

カテゴリ値を0.5などの連続値として解釈しません。ここでも事後平均のランキングを使い、連続獲得関数最適化とは区別します。

In [ ]:
from robotorchan.models import MixedHigherOrderGP, MixedLatentKroneckerGP

mixed_train_category = (train_X[:, :1] > 0.5).to(train_X)
mixed_train_X = torch.cat([train_X, mixed_train_category], dim=-1)
mixed_train_Y = spectrum(train_X, train_T) + 0.1 * mixed_train_category

mixed_hogp = MixedHigherOrderGP(
    train_X=mixed_train_X,
    train_Y=mixed_train_Y,
    cat_dims=[2],
)
mixed_hogp.fit(step_limit=75)
mixed_hogp.eval()

mixed_lkgp = MixedLatentKroneckerGP(
    train_X=mixed_train_X,
    train_T=train_T,
    train_Y=mixed_train_Y,
    cat_dims=[2],
)
mixed_lkgp.fit()
mixed_lkgp.eval()

mixed_candidate_design = torch.rand(32, 2)
mixed_candidate_X = torch.cat(
    [
        torch.cat(
            [
                mixed_candidate_design,
                torch.full(
                    (len(mixed_candidate_design), 1),
                    float(category),
                ),
            ],
            dim=-1,
        )
        for category in (0, 1)
    ],
    dim=0,
)

with torch.no_grad():
    mixed_hogp_scores = mixed_hogp.posterior(
        mixed_candidate_X
    ).mean.reshape(len(mixed_candidate_X), -1).mean(dim=-1)
    mixed_lkgp_candidate_mean, _ = latent_original_moments(
        mixed_lkgp, mixed_candidate_X, train_T
    )
    mixed_lkgp_scores = mixed_lkgp_candidate_mean.mean(dim=-1)

for name, scores in [
    ("MixedHigherOrderGP", mixed_hogp_scores),
    ("MixedLatentKroneckerGP", mixed_lkgp_scores),
]:
    best = int(scores.argmax().item())
    candidate = mixed_candidate_X[best:best + 1]
    observed_spectrum = (
        spectrum(candidate[:, :2], train_T)
        + 0.1 * candidate[:, 2:3]
    )
    observed_utility = observed_spectrum.mean()
    assert candidate.shape == (1, 3)
    assert candidate[0, 2].item() in (0.0, 1.0)
    assert torch.isfinite(observed_utility)
    print(name, "candidate:", candidate)
    print("predicted utility:", scores[best].item())
    print("evaluated utility:", observed_utility.item())


## 10. 追加実装例：候補のスペクトル観測を追加する

構造化出力のBOでは、候補を評価すると**スカラー1点ではなく波長方向の応答ベクトル全体**が追加されます。ここでは既存の `candidate_X` のうち、LatentKroneckerGPが選んだ候補を改めて特定し、学習用の `X` と `Y` に追加します。

計算時間の大きいモデル再学習はこのセルでは実施しません。更新したデータを渡して新しいモデルを構築し、対応する学習処理を実行することが次のステップです。

In [ ]:
best_lkgp_index = lkgp_scores.argmax()
new_X = candidate_X[best_lkgp_index:best_lkgp_index + 1]
new_spectrum = spectrum(new_X, train_T)
updated_X = torch.cat([train_X, new_X], dim=0)
updated_Y = torch.cat([train_Y, new_spectrum], dim=0)

assert new_spectrum.shape == (1, n_wavelengths)
assert updated_X.shape == (n_train + 1, 2)
assert updated_Y.shape == (n_train + 1, n_wavelengths)
print("new process condition:", new_X)
print("new spectrum shape:", new_spectrum.shape)
print("updated structured observations:", updated_Y.shape)